# 🏦 Projet Machine Learning - Prédiction du RiskScore

**Master 2 ISF - Université Paris-Dauphine**  
**Année Scolaire 2025/2026**

---

## 📝 Contexte du Projet

Nous sommes Data Scientists recrutés par une banque internationale pour améliorer le processus d'évaluation du risque de défaut de paiement avant l'octroi d'un prêt.

### Objectif
Développer un modèle de Machine Learning capable de **prédire le RiskScore** (0-100) associé à une demande de prêt.

### Bénéfices Attendus
- ✅ Réduire le taux de défaut en identifiant les profils à risque
- ✅ Optimiser le taux d'acceptation en ciblant les clients fiables  
- ✅ Automatiser et fiabiliser le processus d'octroi de crédit

---

## 📚 Table des Matières

1. [Imports et Configuration](#1-imports)
2. [Chargement et Exploration des Données](#2-exploration)
3. [Data Engineering & Preprocessing](#3-engineering)
4. [Feature Engineering](#4-features)
5. [Modélisation](#5-modelisation)
   - 5.1 Baseline Models
   - 5.2 Modèles Avancés (CART, Random Forest, Gradient Boosting)
   - 5.3 Hyperparamétrage & Cross-Validation
6. [Évaluation et Comparaison](#6-evaluation)
7. [Modèle Final et Recommandations](#7-final)
8. [Conclusion et Limites](#8-conclusion)

---
## 1️⃣ Imports et Configuration <a id='1-imports'></a>

In [ ]:
# Bibliothèques de base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Scikit-learn - Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modèles de régression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Métriques d'évaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

print("✅ Tous les packages sont importés avec succès!")
print(f"Version de pandas: {pd.__version__}")
print(f"Version de scikit-learn: {__import__('sklearn').__version__}")

---
## 2️⃣ Chargement et Exploration des Données <a id='2-exploration'></a>

In [ ]:
# Chargement des données
df = pd.read_csv('/mnt/user-data/uploads/Loan_data.csv')

print("="*80)
print("📊 APERÇU DES DONNÉES")
print("="*80)
print(f"\n📏 Dimensions: {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"\n🎯 Variable cible: RiskScore")
print(f"   • Minimum: {df['RiskScore'].min()}")
print(f"   • Maximum: {df['RiskScore'].max()}")
print(f"   • Moyenne: {df['RiskScore'].mean():.2f}")
print(f"   • Médiane: {df['RiskScore'].median():.2f}")
print(f"   • Écart-type: {df['RiskScore'].std():.2f}")

# Afficher les premières lignes
df.head()

In [ ]:
# Informations sur les colonnes
print("\n📋 INFORMATIONS SUR LES COLONNES\n")
df.info()

In [ ]:
# Statistiques descriptives
print("\n📈 STATISTIQUES DESCRIPTIVES\n")
df.describe()

In [ ]:
# Identification des types de variables
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_features.remove('RiskScore')  # Retirer la variable cible

categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n✅ Variables numériques ({len(numeric_features)}):")
print(numeric_features)

print(f"\n✅ Variables catégorielles ({len(categorical_features)}):")
print(categorical_features)

In [ ]:
# Vérification des valeurs manquantes
missing_values = df.isnull().sum()
missing_percent = 100 * df.isnull().sum() / len(df)
missing_table = pd.concat([missing_values, missing_percent], axis=1, 
                         keys=['Nombre', 'Pourcentage'])
missing_table = missing_table[missing_table['Nombre'] > 0].sort_values('Nombre', ascending=False)

if len(missing_table) > 0:
    print("\n⚠️ VALEURS MANQUANTES DÉTECTÉES:\n")
    print(missing_table)
else:
    print("\n✅ Aucune valeur manquante détectée!")

### 2.1 Analyse de la Variable Cible (RiskScore)

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogramme
axes[0].hist(df['RiskScore'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(df['RiskScore'].mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {df["RiskScore"].mean():.2f}')
axes[0].axvline(df['RiskScore'].median(), color='green', linestyle='--', linewidth=2, label=f'Médiane: {df["RiskScore"].median():.2f}')
axes[0].set_xlabel('RiskScore', fontsize=12)
axes[0].set_ylabel('Fréquence', fontsize=12)
axes[0].set_title('Distribution du RiskScore', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot
axes[1].boxplot(df['RiskScore'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='black'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('RiskScore', fontsize=12)
axes[1].set_title('Boxplot du RiskScore', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# QQ-plot pour normalité
stats.probplot(df['RiskScore'], dist="norm", plot=axes[2])
axes[2].set_title('QQ-Plot (Test de Normalité)', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test de normalité Shapiro-Wilk (sur un échantillon)
sample_size = min(5000, len(df))
sample = df['RiskScore'].sample(sample_size, random_state=42)
stat, p_value = stats.shapiro(sample)
print(f"\n📊 Test de Shapiro-Wilk (sur échantillon de {sample_size}):")
print(f"   • Statistique: {stat:.4f}")
print(f"   • P-value: {p_value:.4f}")
if p_value > 0.05:
    print("   ✅ La distribution semble normale (p > 0.05)")
else:
    print("   ⚠️ La distribution n'est pas normale (p < 0.05)")

### 2.2 Analyse des Variables Catégorielles

In [ ]:
# Analyse des variables catégorielles
print("\n🔍 ANALYSE DES VARIABLES CATÉGORIELLES\n")

for col in categorical_features:
    n_unique = df[col].nunique()
    print(f"\n{col}:")
    print(f"  • Nombre de valeurs uniques: {n_unique}")
    if n_unique <= 10:
        print(f"  • Valeurs: {df[col].unique().tolist()}")
        print(f"  • Distribution:\n{df[col].value_counts()}")
    else:
        print(f"  • Top 5 valeurs:\n{df[col].value_counts().head()}")

In [ ]:
# Visualisation des variables catégorielles vs RiskScore
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, col in enumerate(categorical_features):
    if idx < 6:
        df.boxplot(column='RiskScore', by=col, ax=axes[idx])
        axes[idx].set_title(f'RiskScore par {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('RiskScore')
        plt.sca(axes[idx])
        plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

### 2.3 Matrice de Corrélation

In [ ]:
# Calcul de la matrice de corrélation
correlation_matrix = df[numeric_features + ['RiskScore']].corr()

# Top corrélations avec RiskScore
risk_corr = correlation_matrix['RiskScore'].sort_values(ascending=False)
print("\n📊 TOP 10 CORRÉLATIONS AVEC RISKSCORE:\n")
print(risk_corr.head(11))  # 11 car inclut RiskScore lui-même

# Visualisation de la matrice de corrélation
plt.figure(figsize=(20, 16))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation des Variables Numériques', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation des corrélations les plus fortes avec RiskScore
top_features = risk_corr.head(6).index.tolist()[1:]  # Exclure RiskScore lui-même

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_features):
    axes[idx].scatter(df[feature], df['RiskScore'], alpha=0.3)
    axes[idx].set_xlabel(feature, fontsize=11)
    axes[idx].set_ylabel('RiskScore', fontsize=11)
    axes[idx].set_title(f'{feature} vs RiskScore\n(corr: {risk_corr[feature]:.3f})', fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.4 Détection des Valeurs Aberrantes (Outliers)

In [ ]:
# Fonction pour détecter les outliers avec la méthode IQR
def detect_outliers_iqr(df, columns):
    outliers_dict = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outliers_dict[col] = {
            'count': len(outliers),
            'percentage': 100 * len(outliers) / len(df),
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }
    
    return outliers_dict

# Détecter les outliers
outliers_info = detect_outliers_iqr(df, numeric_features)

# Afficher les variables avec le plus d'outliers
outliers_df = pd.DataFrame(outliers_info).T
outliers_df = outliers_df.sort_values('count', ascending=False)

print("\n🔍 DÉTECTION DES VALEURS ABERRANTES (Méthode IQR)\n")
print(outliers_df.head(10))

---
## 3️⃣ Data Engineering & Preprocessing <a id='3-engineering'></a>

### 3.1 Traitement des Dates

In [ ]:
# Traiter la colonne ApplicationDate si elle existe
if 'ApplicationDate' in df.columns:
    df['ApplicationDate'] = pd.to_datetime(df['ApplicationDate'])
    df['Application_Year'] = df['ApplicationDate'].dt.year
    df['Application_Month'] = df['ApplicationDate'].dt.month
    df['Application_DayOfWeek'] = df['ApplicationDate'].dt.dayofweek
    df['Application_Quarter'] = df['ApplicationDate'].dt.quarter
    
    print("✅ Features temporelles créées:")
    print("   • Application_Year")
    print("   • Application_Month")
    print("   • Application_DayOfWeek")
    print("   • Application_Quarter")
    
    # Supprimer la colonne date originale
    df = df.drop('ApplicationDate', axis=1)
    
    # Mettre à jour les listes de features
    numeric_features.extend(['Application_Year', 'Application_Month', 'Application_DayOfWeek', 'Application_Quarter'])

### 3.2 Encodage des Variables Catégorielles

In [ ]:
# Créer une copie pour le preprocessing
df_processed = df.copy()

# Label Encoding pour les variables binaires ou ordinales
label_encode_cols = []
for col in categorical_features:
    if df[col].nunique() == 2:
        label_encode_cols.append(col)

print(f"\n✅ Variables à encoder avec Label Encoding: {label_encode_cols}")

le_dict = {}
for col in label_encode_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    le_dict[col] = le
    print(f"   • {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# One-Hot Encoding pour les autres variables catégorielles
onehot_cols = [col for col in categorical_features if col not in label_encode_cols]
print(f"\n✅ Variables à encoder avec One-Hot Encoding: {onehot_cols}")

if len(onehot_cols) > 0:
    df_processed = pd.get_dummies(df_processed, columns=onehot_cols, drop_first=True)
    print(f"   • Nouvelles dimensions après One-Hot Encoding: {df_processed.shape}")

---
## 4️⃣ Feature Engineering <a id='4-features'></a>

In [ ]:
# Créer des features supplémentaires pertinentes
print("\n🔧 CRÉATION DE NOUVELLES FEATURES\n")

# 1. Ratio dette/revenu mensuel
if 'MonthlyDebtPayments' in df_processed.columns and 'MonthlyIncome' in df_processed.columns:
    df_processed['DebtToMonthlyIncome'] = df_processed['MonthlyDebtPayments'] / (df_processed['MonthlyIncome'] + 1)
    print("✅ DebtToMonthlyIncome créée")

# 2. Capacité d'épargne mensuelle
if 'MonthlyIncome' in df_processed.columns and 'MonthlyLoanPayment' in df_processed.columns:
    df_processed['MonthlySavingsCapacity'] = df_processed['MonthlyIncome'] - df_processed['MonthlyLoanPayment'] - df_processed['MonthlyDebtPayments']
    print("✅ MonthlySavingsCapacity créée")

# 3. Ratio actifs/passifs
if 'TotalAssets' in df_processed.columns and 'TotalLiabilities' in df_processed.columns:
    df_processed['AssetToLiabilityRatio'] = df_processed['TotalAssets'] / (df_processed['TotalLiabilities'] + 1)
    print("✅ AssetToLiabilityRatio créée")

# 4. Charge mensuelle totale
if 'MonthlyLoanPayment' in df_processed.columns and 'MonthlyDebtPayments' in df_processed.columns:
    df_processed['TotalMonthlyPayments'] = df_processed['MonthlyLoanPayment'] + df_processed['MonthlyDebtPayments']
    print("✅ TotalMonthlyPayments créée")

# 5. Ratio montant prêt / revenu annuel
if 'LoanAmount' in df_processed.columns and 'AnnualIncome' in df_processed.columns:
    df_processed['LoanToIncomeRatio'] = df_processed['LoanAmount'] / (df_processed['AnnualIncome'] + 1)
    print("✅ LoanToIncomeRatio créée")

# 6. Score de solvabilité ajusté par l'âge
if 'CreditScore' in df_processed.columns and 'Age' in df_processed.columns:
    df_processed['CreditScore_Age_Interaction'] = df_processed['CreditScore'] * (df_processed['Age'] / 100)
    print("✅ CreditScore_Age_Interaction créée")

# 7. Indicateur de stabilité financière
if 'JobTenure' in df_processed.columns and 'LengthOfCreditHistory' in df_processed.columns:
    df_processed['FinancialStabilityScore'] = (df_processed['JobTenure'] + df_processed['LengthOfCreditHistory']) / 2
    print("✅ FinancialStabilityScore créée")

print(f"\n📊 Nouvelles dimensions du dataset: {df_processed.shape}")

---
## 5️⃣ Modélisation <a id='5-modelisation'></a>

### 5.1 Préparation des Données (Sampling)

In [ ]:
# Séparer les features et la target
X = df_processed.drop('RiskScore', axis=1)
y = df_processed['RiskScore']

print(f"\n📊 Dimensions finales:")
print(f"   • X (features): {X.shape}")
print(f"   • y (target): {y.shape}")

# Split Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n✅ Split Train/Test effectué:")
print(f"   • Train: {X_train.shape[0]:,} échantillons ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   • Test:  {X_test.shape[0]:,} échantillons ({X_test.shape[0]/len(X)*100:.1f}%)")

In [ ]:
# Standardisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Standardisation effectuée avec StandardScaler")
print(f"   • Moyenne des features (train): {X_train_scaled.mean():.4f}")
print(f"   • Écart-type des features (train): {X_train_scaled.std():.4f}")

### 5.2 Fonction d'Évaluation des Modèles

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name="Model"):
    """
    Évalue un modèle de régression et retourne les métriques.
    
    Parameters:
    -----------
    model : estimator
        Le modèle à évaluer
    X_train, X_test : array-like
        Features d'entraînement et de test
    y_train, y_test : array-like
        Target d'entraînement et de test
    model_name : str
        Nom du modèle pour l'affichage
    
    Returns:
    --------
    dict : Dictionnaire contenant toutes les métriques
    """
    # Prédictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calcul des métriques
    metrics = {
        'Model': model_name,
        'R2_train': r2_score(y_train, y_train_pred),
        'R2_test': r2_score(y_test, y_test_pred),
        'RMSE_train': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'RMSE_test': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'MAE_train': mean_absolute_error(y_train, y_train_pred),
        'MAE_test': mean_absolute_error(y_test, y_test_pred),
        'MAPE_train': mean_absolute_percentage_error(y_train, y_train_pred),
        'MAPE_test': mean_absolute_percentage_error(y_test, y_test_pred)
    }
    
    # Affichage
    print(f"\n{'='*70}")
    print(f"📊 RÉSULTATS - {model_name}")
    print(f"{'='*70}")
    print(f"\n{'Métrique':<20} {'Train':<15} {'Test':<15} {'Écart':<15}")
    print(f"{'-'*70}")
    print(f"{'R² Score':<20} {metrics['R2_train']:<15.4f} {metrics['R2_test']:<15.4f} {abs(metrics['R2_train']-metrics['R2_test']):<15.4f}")
    print(f"{'RMSE':<20} {metrics['RMSE_train']:<15.4f} {metrics['RMSE_test']:<15.4f} {abs(metrics['RMSE_train']-metrics['RMSE_test']):<15.4f}")
    print(f"{'MAE':<20} {metrics['MAE_train']:<15.4f} {metrics['MAE_test']:<15.4f} {abs(metrics['MAE_train']-metrics['MAE_test']):<15.4f}")
    print(f"{'MAPE (%)':<20} {metrics['MAPE_train']*100:<15.2f} {metrics['MAPE_test']*100:<15.2f} {abs(metrics['MAPE_train']-metrics['MAPE_test'])*100:<15.2f}")
    
    # Diagnostic overfitting/underfitting
    r2_diff = metrics['R2_train'] - metrics['R2_test']
    if r2_diff > 0.1:
        print(f"\n⚠️  ATTENTION: Surapprentissage détecté (écart R²: {r2_diff:.4f})")
    elif metrics['R2_test'] < 0.5:
        print(f"\n⚠️  ATTENTION: Sous-apprentissage possible (R² test: {metrics['R2_test']:.4f})")
    else:
        print(f"\n✅ Modèle équilibré")
    
    return metrics

# Liste pour stocker les résultats
results_list = []

print("\n✅ Fonction d'évaluation définie")

### 5.3 Modèles Baseline

In [ ]:
print("\n" + "="*70)
print("🎯 ENTRAÎNEMENT DES MODÈLES BASELINE")
print("="*70)

In [ ]:
# 1. Régression Linéaire
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
results_list.append(evaluate_model(lr, X_train_scaled, X_test_scaled, y_train, y_test, "Linear Regression"))

In [ ]:
# 2. Ridge Regression
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)
results_list.append(evaluate_model(ridge, X_train_scaled, X_test_scaled, y_train, y_test, "Ridge Regression"))

In [ ]:
# 3. Lasso Regression
lasso = Lasso(alpha=0.1, random_state=42)
lasso.fit(X_train_scaled, y_train)
results_list.append(evaluate_model(lasso, X_train_scaled, X_test_scaled, y_train, y_test, "Lasso Regression"))

### 5.4 Modèles Basés sur les Arbres (CART)

In [ ]:
print("\n" + "="*70)
print("🌳 MODÈLES BASÉS SUR LES ARBRES")
print("="*70)

In [ ]:
# 4. Decision Tree (CART)
dt = DecisionTreeRegressor(max_depth=10, min_samples_split=20, min_samples_leaf=10, random_state=42)
dt.fit(X_train, y_train)  # Les arbres n'ont pas besoin de standardisation
results_list.append(evaluate_model(dt, X_train, X_test, y_train, y_test, "Decision Tree (CART)"))

### 5.5 Modèles Ensemble

In [ ]:
print("\n" + "="*70)
print("🎭 MODÈLES ENSEMBLE (BAGGING & BOOSTING)")
print("="*70)

In [ ]:
# 5. Random Forest
print("\n⏳ Entraînement du Random Forest en cours...")
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
results_list.append(evaluate_model(rf, X_train, X_test, y_train, y_test, "Random Forest"))

In [ ]:
# 6. Gradient Boosting
print("\n⏳ Entraînement du Gradient Boosting en cours...")
gbm = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
gbm.fit(X_train, y_train)
results_list.append(evaluate_model(gbm, X_train, X_test, y_train, y_test, "Gradient Boosting"))

### 5.6 Cross-Validation sur les Meilleurs Modèles

In [ ]:
print("\n" + "="*70)
print("🔄 CROSS-VALIDATION (K-FOLD avec k=5)")
print("="*70)

# Définir les modèles à valider
models_cv = {
    'Random Forest': rf,
    'Gradient Boosting': gbm,
    'Decision Tree': dt
}

cv_results = {}
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models_cv.items():
    print(f"\n⏳ Cross-validation de {name}...")
    
    # Scores R²
    cv_scores_r2 = cross_val_score(model, X_train, y_train, cv=kfold, 
                                    scoring='r2', n_jobs=-1)
    
    # Scores RMSE (négatif dans sklearn)
    cv_scores_rmse = -cross_val_score(model, X_train, y_train, cv=kfold, 
                                       scoring='neg_root_mean_squared_error', n_jobs=-1)
    
    cv_results[name] = {
        'R2_mean': cv_scores_r2.mean(),
        'R2_std': cv_scores_r2.std(),
        'RMSE_mean': cv_scores_rmse.mean(),
        'RMSE_std': cv_scores_rmse.std()
    }
    
    print(f"   R² Score:  {cv_scores_r2.mean():.4f} (+/- {cv_scores_r2.std() * 2:.4f})")
    print(f"   RMSE:      {cv_scores_rmse.mean():.4f} (+/- {cv_scores_rmse.std() * 2:.4f})")

print("\n✅ Cross-validation terminée")

### 5.7 Hyperparamétrage (Grid Search) - Random Forest

In [ ]:
print("\n" + "="*70)
print("🎛️ HYPERPARAMÉTRAGE - RANDOM FOREST (Grid Search)")
print("="*70)

# Définir la grille de paramètres
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 5]
}

print("\n⏳ Grid Search en cours (cela peut prendre plusieurs minutes)...")
print(f"   Nombre total de combinaisons: {np.prod([len(v) for v in param_grid_rf.values()])}")

grid_search_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

print(f"\n✅ Grid Search terminé!")
print(f"\n🏆 Meilleurs paramètres:")
for param, value in grid_search_rf.best_params_.items():
    print(f"   • {param}: {value}")

print(f"\n📊 Meilleur score R² (CV): {grid_search_rf.best_score_:.4f}")

# Évaluer le meilleur modèle
best_rf = grid_search_rf.best_estimator_
results_list.append(evaluate_model(best_rf, X_train, X_test, y_train, y_test, "Random Forest (Optimisé)"))

### 5.8 Hyperparamétrage (Grid Search) - Gradient Boosting

In [ ]:
print("\n" + "="*70)
print("🎛️ HYPERPARAMÉTRAGE - GRADIENT BOOSTING (Grid Search)")
print("="*70)

# Définir la grille de paramètres
param_grid_gbm = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [10, 20]
}

print("\n⏳ Grid Search en cours (cela peut prendre plusieurs minutes)...")
print(f"   Nombre total de combinaisons: {np.prod([len(v) for v in param_grid_gbm.values()])}")

grid_search_gbm = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid_gbm,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search_gbm.fit(X_train, y_train)

print(f"\n✅ Grid Search terminé!")
print(f"\n🏆 Meilleurs paramètres:")
for param, value in grid_search_gbm.best_params_.items():
    print(f"   • {param}: {value}")

print(f"\n📊 Meilleur score R² (CV): {grid_search_gbm.best_score_:.4f}")

# Évaluer le meilleur modèle
best_gbm = grid_search_gbm.best_estimator_
results_list.append(evaluate_model(best_gbm, X_train, X_test, y_train, y_test, "Gradient Boosting (Optimisé)"))

---
## 6️⃣ Évaluation et Comparaison des Modèles <a id='6-evaluation'></a>

In [ ]:
# Créer un DataFrame avec tous les résultats
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('R2_test', ascending=False)

print("\n" + "="*100)
print("📊 TABLEAU RÉCAPITULATIF DES PERFORMANCES")
print("="*100)
print(results_df.to_string(index=False))

# Identifier le meilleur modèle
best_model_name = results_df.iloc[0]['Model']
best_r2 = results_df.iloc[0]['R2_test']
print(f"\n🏆 MEILLEUR MODÈLE: {best_model_name}")
print(f"   R² Score Test: {best_r2:.4f}")

In [ ]:
# Visualisation comparative des modèles
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. R² Score
ax1 = axes[0, 0]
x_pos = np.arange(len(results_df))
ax1.barh(x_pos, results_df['R2_test'], color='skyblue', label='Test')
ax1.barh(x_pos, results_df['R2_train'], color='lightcoral', alpha=0.6, label='Train')
ax1.set_yticks(x_pos)
ax1.set_yticklabels(results_df['Model'])
ax1.set_xlabel('R² Score', fontsize=12)
ax1.set_title('Comparaison des R² Scores', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='x')

# 2. RMSE
ax2 = axes[0, 1]
ax2.barh(x_pos, results_df['RMSE_test'], color='lightgreen', label='Test')
ax2.barh(x_pos, results_df['RMSE_train'], color='salmon', alpha=0.6, label='Train')
ax2.set_yticks(x_pos)
ax2.set_yticklabels(results_df['Model'])
ax2.set_xlabel('RMSE', fontsize=12)
ax2.set_title('Comparaison des RMSE', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

# 3. MAE
ax3 = axes[1, 0]
ax3.barh(x_pos, results_df['MAE_test'], color='gold', label='Test')
ax3.barh(x_pos, results_df['MAE_train'], color='orange', alpha=0.6, label='Train')
ax3.set_yticks(x_pos)
ax3.set_yticklabels(results_df['Model'])
ax3.set_xlabel('MAE', fontsize=12)
ax3.set_title('Comparaison des MAE', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

# 4. MAPE
ax4 = axes[1, 1]
ax4.barh(x_pos, results_df['MAPE_test']*100, color='plum', label='Test')
ax4.barh(x_pos, results_df['MAPE_train']*100, color='orchid', alpha=0.6, label='Train')
ax4.set_yticks(x_pos)
ax4.set_yticklabels(results_df['Model'])
ax4.set_xlabel('MAPE (%)', fontsize=12)
ax4.set_title('Comparaison des MAPE', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

---
## 7️⃣ Analyse du Modèle Final <a id='7-final'></a>

In [ ]:
# Sélectionner le meilleur modèle basé sur le R² test
if 'Optimisé' in best_model_name:
    if 'Random Forest' in best_model_name:
        final_model = best_rf
    else:
        final_model = best_gbm
else:
    # Sélectionner parmi les modèles non optimisés
    models_dict = {
        'Random Forest': rf,
        'Gradient Boosting': gbm,
        'Decision Tree (CART)': dt
    }
    final_model = models_dict.get(best_model_name, gbm)

print(f"\n🎯 Modèle final sélectionné: {best_model_name}")

### 7.1 Importance des Features

In [ ]:
# Feature Importance
if hasattr(final_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': final_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n📊 TOP 20 FEATURES LES PLUS IMPORTANTES:\n")
    print(feature_importance.head(20).to_string(index=False))
    
    # Visualisation
    plt.figure(figsize=(12, 8))
    top_n = 20
    plt.barh(range(top_n), feature_importance['Importance'].head(top_n), color='steelblue')
    plt.yticks(range(top_n), feature_importance['Feature'].head(top_n))
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Top {top_n} Features les Plus Importantes - {best_model_name}', 
              fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️ Le modèle sélectionné ne supporte pas l'analyse d'importance des features.")

### 7.2 Analyse des Prédictions

In [ ]:
# Prédictions finales
y_train_pred_final = final_model.predict(X_train)
y_test_pred_final = final_model.predict(X_test)

# Visualisation des prédictions vs réalité
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Train
axes[0].scatter(y_train, y_train_pred_final, alpha=0.3, s=20)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'r--', lw=2, label='Prédiction parfaite')
axes[0].set_xlabel('RiskScore Réel', fontsize=12)
axes[0].set_ylabel('RiskScore Prédit', fontsize=12)
axes[0].set_title(f'Prédictions vs Réalité (Train) - {best_model_name}', 
                  fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test
axes[1].scatter(y_test, y_test_pred_final, alpha=0.3, s=20, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Prédiction parfaite')
axes[1].set_xlabel('RiskScore Réel', fontsize=12)
axes[1].set_ylabel('RiskScore Prédit', fontsize=12)
axes[1].set_title(f'Prédictions vs Réalité (Test) - {best_model_name}', 
                  fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.3 Analyse des Résidus

In [ ]:
# Calcul des résidus
residuals_train = y_train - y_train_pred_final
residuals_test = y_test - y_test_pred_final

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribution des résidus (Train)
axes[0, 0].hist(residuals_train, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Résidus', fontsize=12)
axes[0, 0].set_ylabel('Fréquence', fontsize=12)
axes[0, 0].set_title('Distribution des Résidus (Train)', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# 2. Distribution des résidus (Test)
axes[0, 1].hist(residuals_test, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Résidus', fontsize=12)
axes[0, 1].set_ylabel('Fréquence', fontsize=12)
axes[0, 1].set_title('Distribution des Résidus (Test)', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Résidus vs Prédictions (Train)
axes[1, 0].scatter(y_train_pred_final, residuals_train, alpha=0.3, s=20)
axes[1, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Prédictions', fontsize=12)
axes[1, 0].set_ylabel('Résidus', fontsize=12)
axes[1, 0].set_title('Résidus vs Prédictions (Train)', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. Résidus vs Prédictions (Test)
axes[1, 1].scatter(y_test_pred_final, residuals_test, alpha=0.3, s=20, color='orange')
axes[1, 1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Prédictions', fontsize=12)
axes[1, 1].set_ylabel('Résidus', fontsize=12)
axes[1, 1].set_title('Résidus vs Prédictions (Test)', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistiques des résidus
print("\n📊 STATISTIQUES DES RÉSIDUS:\n")
print(f"{'Métrique':<25} {'Train':<15} {'Test':<15}")
print("-" * 55)
print(f"{'Moyenne':<25} {residuals_train.mean():<15.4f} {residuals_test.mean():<15.4f}")
print(f"{'Écart-type':<25} {residuals_train.std():<15.4f} {residuals_test.std():<15.4f}")
print(f"{'Min':<25} {residuals_train.min():<15.4f} {residuals_test.min():<15.4f}")
print(f"{'Max':<25} {residuals_train.max():<15.4f} {residuals_test.max():<15.4f}")

---
## 8️⃣ Conclusion et Recommandations <a id='8-conclusion'></a>

### 8.1 Synthèse des Résultats

In [ ]:
print("\n" + "="*70)
print("📋 SYNTHÈSE FINALE DU PROJET")
print("="*70)

print(f"\n🎯 Objectif: Prédire le RiskScore (0-100) pour évaluer le risque de défaut")
print(f"\n📊 Données:")
print(f"   • {len(df):,} observations")
print(f"   • {len(X.columns)} features après preprocessing")
print(f"   • {len(numeric_features)} variables numériques originales")
print(f"   • {len(categorical_features)} variables catégorielles originales")

print(f"\n🤖 Modèles testés: {len(results_df)}")
for idx, row in results_df.iterrows():
    print(f"   • {row['Model']}: R²={row['R2_test']:.4f}, RMSE={row['RMSE_test']:.4f}")

print(f"\n🏆 Meilleur modèle: {best_model_name}")
best_results = results_df.iloc[0]
print(f"   • R² Score (test): {best_results['R2_test']:.4f}")
print(f"   • RMSE (test): {best_results['RMSE_test']:.4f}")
print(f"   • MAE (test): {best_results['MAE_test']:.4f}")
print(f"   • MAPE (test): {best_results['MAPE_test']*100:.2f}%")

print("\n✅ Le modèle explique {:.1f}% de la variance du RiskScore".format(best_results['R2_test']*100))

### 8.2 Recommandations pour la Mise en Production

#### 📌 Recommandations Opérationnelles

1. **Déploiement du Modèle**
   - Intégrer le modèle dans le système d'évaluation des demandes de prêt
   - Créer une API REST pour les prédictions en temps réel
   - Mettre en place un système de monitoring des performances

2. **Seuils de Décision**
   - RiskScore < 40: Faible risque → Approbation automatique
   - RiskScore 40-60: Risque modéré → Revue manuelle recommandée
   - RiskScore > 60: Risque élevé → Refus ou conditions strictes

3. **Amélioration Continue**
   - Réentraîner le modèle tous les 3-6 mois avec nouvelles données
   - Monitorer les dérives de performance (data drift)
   - Collecter feedback sur les décisions pour améliorer le modèle

4. **Explicabilité**
   - Utiliser SHAP values pour expliquer les prédictions individuelles
   - Fournir aux analystes les features les plus influentes
   - Documenter les limites du modèle pour usage éthique

### 8.3 Limites et Axes d'Amélioration

#### ⚠️ Limites Identifiées

1. **Données**
   - Période de collecte limitée → Peut ne pas capturer tous les cycles économiques
   - Absence de variables contextuelles (conditions économiques macro)
   - Possibles biais dans les données historiques

2. **Modélisation**
   - Modèle statique → Ne s'adapte pas automatiquement aux changements
   - Features engineering manuel → Peut manquer interactions complexes
   - Hyperparamétrage limité par ressources computationnelles

3. **Évaluation**
   - Métriques statistiques uniquement → Impact business à quantifier
   - Pas de test A/B en conditions réelles
   - Analyse limitée des cas extrêmes

#### 🚀 Axes d'Amélioration

1. **Court terme**
   - Tester d'autres algorithmes (XGBoost, LightGBM, CatBoost)
   - Approfondir le feature engineering (interactions, polynômes)
   - Optimiser davantage les hyperparamètres

2. **Moyen terme**
   - Collecter données supplémentaires (historique paiements, données externes)
   - Développer modèles spécifiques par segments clients
   - Implémenter système de détection des fraudes

3. **Long terme**
   - Modèle dynamique avec apprentissage continu
   - Intégration de données alternatives (réseaux sociaux, transactions)
   - Deep Learning pour capturer patterns complexes

### 8.4 Sauvegarde du Modèle Final

In [ ]:
import joblib
from datetime import datetime

# Créer un nom de fichier avec timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_filename = f'/mnt/user-data/outputs/best_model_{timestamp}.pkl'
scaler_filename = f'/mnt/user-data/outputs/scaler_{timestamp}.pkl'

# Sauvegarder le modèle et le scaler
joblib.dump(final_model, model_filename)
joblib.dump(scaler, scaler_filename)

print(f"\n✅ Modèle sauvegardé: {model_filename}")
print(f"✅ Scaler sauvegardé: {scaler_filename}")

# Sauvegarder aussi les résultats
results_filename = f'/mnt/user-data/outputs/results_{timestamp}.csv'
results_df.to_csv(results_filename, index=False)
print(f"✅ Résultats sauvegardés: {results_filename}")

---
## 🎉 FIN DU PROJET

Ce notebook présente une approche complète de modélisation Machine Learning pour la prédiction du RiskScore.

**Points clés:**
- ✅ Exploration approfondie des données
- ✅ Preprocessing et feature engineering rigoureux
- ✅ Comparaison de multiples algorithmes
- ✅ Optimisation des hyperparamètres
- ✅ Évaluation complète avec métriques multiples
- ✅ Analyse détaillée du modèle final
- ✅ Recommandations opérationnelles

**Prochaines étapes suggérées:**
1. Tester XGBoost/LightGBM/CatBoost pour comparaison
2. Créer features d'interaction plus sophistiquées
3. Développer une API de prédiction
4. Mettre en place monitoring en production

---
*Projet réalisé dans le cadre du Master 2 ISF - Université Paris-Dauphine*